# 01 - Synthetic event generator (Kinesis-shaped)

Generates **synthetic** behavioral streams for a cross-border remittance app and lands them as JSON-lines files in the UC Volume `raw_events`, one folder per "stream":

| Stream | Folder | Payload highlights |
| --- | --- | --- |
| login | `raw_events/login/` | device_id, ip, geo_country, success, auth_method |
| otp | `raw_events/otp/` | phone, otp_channel, status (sent/failed/verified), attempt_no |
| screen_view | `raw_events/screen_view/` | session_id, screen_name, dwell_ms, screen_seq |
| transfers | `raw_events/transfers/` | transfer_id, amount_usd, corridor, payout_method, device_id, ts, **is_fraud** (label) |

Every record uses a Kinesis envelope: `{partitionKey, sequenceNumber, approximateArrivalTimestamp, data:{...}}`.

**Population:** ~5K customers, ~2% are "fraudster" accounts (account-takeover victims / mule-controlled) that, during incidents, show: new device + foreign IP, failed-login and failed-OTP bursts, rapid screen hops (sub-second dwell), then high-amount transfers to a corridor the customer never used, via cash pickup / wallet. ~25% of incidents are *partial* (weaker signal) and ~1% of legit sessions are *hard negatives* (traveling customer with new device, OTP retries, larger amounts) so the model is not trivially separable.

**Data-quality noise (on purpose):** ~0.5% duplicate records with the same `sequenceNumber` (consumer replay) and ~0.2% malformed records (null customer_id / event_ts, negative amount) to exercise silver dedup + expectations.

**Modes:** one-time 30-day `backfill` (idempotent via a marker file), then a `live` loop writing a micro-batch every `live_interval_seconds` for `live_duration_seconds`. No real customer data; phone numbers use the fictional 555 range and IPs the CGNAT range.

In [0]:
dbutils.widgets.text("catalog", "mlo_serverless_stable_catalog")
dbutils.widgets.text("schema", "fraud_rt")
dbutils.widgets.text("volume", "raw_events")
dbutils.widgets.dropdown("mode", "backfill_and_live", ["backfill", "live", "backfill_and_live"])
dbutils.widgets.text("n_customers", "5000")
dbutils.widgets.text("fraud_rate", "0.02")
dbutils.widgets.text("backfill_days", "30")
dbutils.widgets.text("live_duration_seconds", "60")
dbutils.widgets.text("live_interval_seconds", "5")
dbutils.widgets.text("live_sessions_per_tick", "40")
dbutils.widgets.text("seed", "42")
# Live runs need their own randomness or they replay the same transfer_ids/sessions as earlier runs.
# Empty = fresh seed per run (printed so a run can be reproduced by pasting it back here).
dbutils.widgets.text("live_seed", "")
dbutils.widgets.dropdown("force_backfill", "false", ["true", "false"])

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = dbutils.widgets.get("volume")
MODE = dbutils.widgets.get("mode")
N_CUSTOMERS = int(dbutils.widgets.get("n_customers"))
FRAUD_RATE = float(dbutils.widgets.get("fraud_rate"))
BACKFILL_DAYS = int(dbutils.widgets.get("backfill_days"))
LIVE_DURATION_S = int(dbutils.widgets.get("live_duration_seconds"))
LIVE_INTERVAL_S = int(dbutils.widgets.get("live_interval_seconds"))
LIVE_SESSIONS_PER_TICK = int(dbutils.widgets.get("live_sessions_per_tick"))
SEED = int(dbutils.widgets.get("seed"))  # customer universe + backfill: fixed, reproducible
_live_seed = dbutils.widgets.get("live_seed").strip()
import secrets
LIVE_SEED = int(_live_seed) if _live_seed else secrets.randbits(63)
FORCE_BACKFILL = dbutils.widgets.get("force_backfill") == "true"

VOL_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
STREAMS = ["login", "otp", "screen_view", "transfers"]
DUP_RATE = 0.005   # same sequenceNumber delivered twice (consumer replay)
BAD_RATE = 0.002   # malformed payloads for expectations
print(VOL_ROOT, MODE, N_CUSTOMERS, FRAUD_RATE, "| seed", SEED, "| live_seed", LIVE_SEED)

In [0]:
import json, os, shutil, time, uuid
from collections import defaultdict
from datetime import datetime, timedelta, timezone
import numpy as np

for d in STREAMS + ["_staging", "_meta"]:
    os.makedirs(f"{VOL_ROOT}/{d}", exist_ok=True)

CORRIDORS = ["US-MX", "US-PH", "US-IN", "US-GT", "US-NG"]
CORRIDOR_P = [0.45, 0.20, 0.20, 0.08, 0.07]
PAYOUTS = ["bank_deposit", "cash_pickup", "mobile_wallet", "home_delivery"]
PAYOUT_P = [0.55, 0.25, 0.15, 0.05]
FOREIGN_GEOS = ["RO", "NG", "VN", "BR", "UA", "ID", "RU"]
APP_VERSIONS = ["5.12.0", "5.12.1", "5.13.0", "5.14.2"]
LEGIT_BROWSE = ["history", "rates", "profile", "notifications", "help", "recipients"]
TRANSFER_PATH = ["send_money", "recipient_select", "amount_entry", "review", "confirm"]
ATO_SCREENS = ["profile", "security_settings", "change_phone", "recipients", "add_recipient", "limits",
               "send_money", "amount_entry", "review", "confirm", "rates", "history"]

rng = np.random.default_rng(SEED)

def uid():
    return str(uuid.UUID(bytes=rng.bytes(16), version=4))

def new_device():
    return "dev-" + rng.bytes(6).hex()

def rand_ip():
    return f"100.{rng.integers(64, 128)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}"

def iso(ts):
    return ts.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

def sec(x):
    return timedelta(seconds=float(x))

def make_customers(n, fraud_rate, seed):
    """Deterministic customer universe so backfill and live runs share the same customers."""
    r = np.random.default_rng(seed + 1)
    fraud_idx = set(r.choice(n, int(round(n * fraud_rate)), replace=False).tolist())
    today = datetime.now(timezone.utc).date()
    custs = []
    for i in range(n):
        custs.append({
            "customer_id": f"C{i + 1:06d}",
            "home_corridor": str(r.choice(CORRIDORS, p=CORRIDOR_P)),
            "preferred_payout": str(r.choice(PAYOUTS, p=PAYOUT_P)),
            "typical_amount": float(np.round(np.exp(r.normal(5.5, 0.6)), 2)),  # median ~ $245
            "devices": ["dev-" + r.bytes(6).hex() for _ in range(int(r.choice([1, 2], p=[0.75, 0.25])))],
            "home_ip": f"100.{r.integers(64, 128)}.{r.integers(0, 256)}.{r.integers(1, 255)}",
            "phone": f"+1-555-{r.integers(100, 1000)}-{r.integers(1000, 10000)}",
            "sessions_per_day": float(r.gamma(2.0, 0.4)),
            "transfer_prob": float(r.uniform(0.15, 0.4)),
            "signup_date": str(today - timedelta(days=int(r.integers(60, 1500)))),
            "is_fraudster": i in fraud_idx,
        })
    return custs

customers = make_customers(N_CUSTOMERS, FRAUD_RATE, SEED)
fraudsters = [c for c in customers if c["is_fraudster"]]
print(f"customers={len(customers)}  fraudster accounts={len(fraudsters)} ({len(fraudsters)/len(customers):.2%})")

In [0]:
def _e(stream, ts, c, **kw):
    d = {"event_id": uid(), "customer_id": c["customer_id"], "event_ts": iso(ts)}
    d.update(kw)
    return (stream, ts, d)

def _transfer(ts, c, amount, corridor, payout, recipient_id, device, ip, is_fraud):
    return ("transfers", ts, {
        "transfer_id": uid(), "customer_id": c["customer_id"], "amount_usd": round(float(amount), 2),
        "currency": "USD", "corridor": corridor, "payout_method": payout, "recipient_id": recipient_id,
        "device_id": device, "ip": ip, "ts": iso(ts), "is_fraud": int(is_fraud)})

def legit_session(c, t, hard=False):
    """Normal session. hard=True -> traveling customer (new device, foreign geo, OTP retries, bigger amount)."""
    out, sid = [], uid()
    if hard or rng.random() < 0.01:
        dev = new_device()
    else:
        dev = c["devices"][0] if rng.random() < 0.85 else str(rng.choice(c["devices"]))
    new_dev = dev not in c["devices"]
    ip = rand_ip() if (hard or rng.random() < 0.15) else c["home_ip"]
    geo = str(rng.choice(["MX", "PH", "IN", "CA"])) if hard else "US"
    app = str(rng.choice(APP_VERSIONS))
    base = dict(device_id=dev, ip=ip, geo_country=geo, app_version=app, session_id=sid)
    if rng.random() < 0.05:
        out.append(_e("login", t, c, auth_method="password", success=False, failure_reason="bad_password", **base))
        t += sec(rng.uniform(5, 40))
    out.append(_e("login", t, c, auth_method="biometric" if (rng.random() < 0.6 and not new_dev) else "password",
                  success=True, failure_reason=None, **base))
    t += sec(rng.uniform(1, 3))
    will_transfer = rng.random() < c["transfer_prob"]
    if new_dev or (will_transfer and rng.random() < 0.4):
        n_fail = int(rng.choice([0, 1, 2], p=[0.5, 0.35, 0.15] if hard else [0.85, 0.12, 0.03]))
        otp = dict(device_id=dev, phone=c["phone"], otp_channel="sms" if new_dev else str(rng.choice(["sms", "push"])), session_id=sid)
        out.append(_e("otp", t, c, status="sent", attempt_no=1, **otp))
        for k in range(n_fail):
            t += sec(rng.uniform(15, 60))
            out.append(_e("otp", t, c, status="failed", attempt_no=k + 1, **otp))
        t += sec(rng.uniform(10, 45))
        out.append(_e("otp", t, c, status="verified", attempt_no=n_fail + 1, **otp))
    screens = ["home"] + list(rng.choice(LEGIT_BROWSE, size=int(rng.integers(1, 5)))) + (TRANSFER_PATH if will_transfer else [])
    for i, s in enumerate(screens):
        dwell = int(np.exp(rng.normal(8.8, 0.6)))  # ~6.6s median
        out.append(_e("screen_view", t, c, device_id=dev, session_id=sid, screen_name=str(s), dwell_ms=dwell, screen_seq=i + 1))
        t += timedelta(milliseconds=dwell)
    if will_transfer:
        corridor = c["home_corridor"] if rng.random() < 0.95 else str(rng.choice(CORRIDORS))
        amt = c["typical_amount"] * np.exp(rng.normal(0, 0.35)) * (rng.uniform(1.5, 3.0) if hard else 1.0)
        payout = c["preferred_payout"] if rng.random() < 0.85 else str(rng.choice(PAYOUTS, p=PAYOUT_P))
        recipient = f"R{c['customer_id'][1:]}-{int(rng.integers(1, 4))}"
        out.append(_transfer(t, c, min(amt, 9999), corridor, payout, recipient, dev, ip, 0))
    return out

def ato_session(c, t):
    """Account-takeover session. ~25% partial (weaker signal)."""
    partial = rng.random() < 0.25
    out, sid, dev, ip = [], uid(), new_device(), rand_ip()
    base = dict(device_id=dev, ip=ip, geo_country=str(rng.choice(FOREIGN_GEOS)), app_version=str(rng.choice(APP_VERSIONS)), session_id=sid)
    for _ in range(int(rng.integers(1, 4)) if not partial else int(rng.integers(0, 2))):
        out.append(_e("login", t, c, auth_method="password", success=False, failure_reason="bad_password", **base))
        t += sec(rng.uniform(3, 15))
    out.append(_e("login", t, c, auth_method="password", success=True, failure_reason=None, **base))
    t += sec(rng.uniform(1, 3))
    otp = dict(device_id=dev, phone=c["phone"], otp_channel="sms", session_id=sid)
    n_fail = int(rng.integers(3, 8)) if not partial else int(rng.integers(0, 2))
    out.append(_e("otp", t, c, status="sent", attempt_no=1, **otp))
    for k in range(n_fail):
        t += sec(rng.uniform(5, 25))
        out.append(_e("otp", t, c, status="failed", attempt_no=k + 1, **otp))
    t += sec(rng.uniform(5, 20))
    out.append(_e("otp", t, c, status="verified", attempt_no=n_fail + 1, **otp))
    n_screens = int(rng.integers(10, 26)) if not partial else int(rng.integers(5, 10))
    for i in range(n_screens):
        dwell = int(np.exp(rng.normal(6.6, 0.5)))  # ~735ms median: rapid hops
        out.append(_e("screen_view", t, c, device_id=dev, session_id=sid, screen_name=str(rng.choice(ATO_SCREENS)), dwell_ms=dwell, screen_seq=i + 1))
        t += timedelta(milliseconds=dwell)
    other = [k for k in CORRIDORS if k != c["home_corridor"]]
    for _ in range(int(rng.choice([1, 2], p=[0.7, 0.3]))):
        amt = float(np.clip(c["typical_amount"] * rng.uniform(4, 15), 800, 9500))
        out.append(_transfer(t, c, amt, str(rng.choice(other)), str(rng.choice(["cash_pickup", "mobile_wallet"])),
                             "R" + rng.bytes(4).hex(), dev, ip, 1))
        t += sec(rng.uniform(30, 120))
    return out

print(len(legit_session(customers[0], datetime.now(timezone.utc))), len(ato_session(fraudsters[0], datetime.now(timezone.utc))))

In [0]:
RUN_PREFIX = f"{int(time.time() * 1e6):016d}"  # unique per run -> sequence numbers never collide across runs
_seq = {"n": 0}

def next_seq():
    _seq["n"] += 1
    return f"49{RUN_PREFIX}{_seq['n']:012d}"

def corrupt(stream, p):
    p = dict(p)
    choice = rng.integers(0, 3)
    if choice == 0:
        p["customer_id"] = None
    elif choice == 1:
        p["ts" if stream == "transfers" else "event_ts"] = None
    elif stream == "transfers":
        p["amount_usd"] = -abs(p["amount_usd"])
    else:
        p["event_ts"] = "not-a-timestamp"
    return p

def to_record(ts, payload):
    lag = rng.exponential(0.4) if rng.random() > 0.01 else rng.uniform(30, 600)  # 1% late arrivals
    return {"partitionKey": payload["customer_id"] or "unknown", "sequenceNumber": next_seq(),
            "approximateArrivalTimestamp": round(ts.timestamp() + lag, 3), "data": payload}

def write_batch(events, tag):
    """events: list of (stream, ts, payload). Writes one JSONL file per stream; staged then moved so Auto Loader never sees partial files."""
    by_stream = defaultdict(list)
    for stream, ts, p in events:
        if rng.random() < BAD_RATE:
            p = corrupt(stream, p)
        rec = to_record(ts, p)
        by_stream[stream].append(rec)
        if rng.random() < DUP_RATE:
            by_stream[stream].append(dict(rec))
    stats = {}
    for stream, recs in by_stream.items():
        recs.sort(key=lambda r: r["approximateArrivalTimestamp"])
        fname = f"{tag}_{uuid.uuid4().hex[:8]}.json"
        tmp = f"{VOL_ROOT}/_staging/{stream}_{fname}"
        with open(tmp, "w") as f:
            f.write("\n".join(json.dumps(r, separators=(",", ":")) for r in recs) + "\n")
        shutil.move(tmp, f"{VOL_ROOT}/{stream}/{fname}")
        stats[stream] = len(recs)
    return stats

In [0]:
from pyspark.sql import functions as F

cust_rows = [(c["customer_id"], c["home_corridor"], c["preferred_payout"], c["phone"], c["devices"][0], c["home_ip"], c["signup_date"])
             for c in customers]
cust_df = (spark.createDataFrame(cust_rows, "customer_id string, home_corridor string, preferred_payout string, phone string, primary_device_id string, home_ip string, signup_date string")
           .withColumn("signup_date", F.to_date("signup_date")))
# Ground-truth fraudster flag is intentionally NOT written here (label only lives on transfers.is_fraud) to avoid leakage.
cust_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.{SCHEMA}.customers")
display(spark.table(f"{CATALOG}.{SCHEMA}.customers").limit(5))

In [0]:
MARKER = f"{VOL_ROOT}/_meta/backfill_done.json"
HOUR_P = np.array([1, 1, 1, 1, 1, 2, 3, 4, 5, 5, 5, 5, 6, 6, 5, 5, 5, 6, 8, 9, 9, 8, 5, 2], dtype=float)
HOUR_P /= HOUR_P.sum()

def run_backfill(days):
    now = datetime.now(timezone.utc).replace(microsecond=0)
    start = (now - timedelta(days=days)).replace(hour=0, minute=0, second=0)
    # Pre-schedule 1-3 ATO incidents per fraudster across the window (skewed to recent weeks).
    incidents = defaultdict(list)
    for c in fraudsters:
        for _ in range(int(rng.integers(1, 4))):
            d = int(min(days - 1, rng.triangular(0, days * 0.7, days - 1)))
            incidents[d].append(c)
    totals = defaultdict(int)
    for d in range(days):
        day0 = start + timedelta(days=d)
        evs = []
        for c in customers:
            for _ in range(rng.poisson(c["sessions_per_day"])):
                t = day0 + timedelta(hours=int(rng.choice(24, p=HOUR_P)), seconds=float(rng.uniform(0, 3600)))
                evs += legit_session(c, t, hard=rng.random() < 0.01)
        for c in incidents[d]:
            t = day0 + timedelta(hours=int(rng.integers(0, 24)), seconds=float(rng.uniform(0, 3600)))
            evs += ato_session(c, t)
        evs = [e for e in evs if e[1] <= now]
        for k, v in write_batch(evs, f"backfill_{day0:%Y%m%d}").items():
            totals[k] += v
    with open(MARKER, "w") as f:
        json.dump({"completed_at": iso(datetime.now(timezone.utc)), "days": days, "seed": SEED, "records": totals,
                   "incidents": sum(len(v) for v in incidents.values())}, f)
    return dict(totals), sum(len(v) for v in incidents.values())

if MODE in ("backfill", "backfill_and_live"):
    if os.path.exists(MARKER) and not FORCE_BACKFILL:
        print("Backfill already done, skipping:", open(MARKER).read())
    else:
        t0 = time.time()
        totals, n_inc = run_backfill(BACKFILL_DAYS)
        print(f"Backfill wrote {totals} records, {n_inc} ATO incidents in {time.time() - t0:.0f}s")

In [0]:
def live_tick(now):
    weights = np.array([c["sessions_per_day"] for c in customers]); weights /= weights.sum()
    evs = []
    picks = rng.choice(len(customers), size=rng.poisson(LIVE_SESSIONS_PER_TICK), p=weights)
    sessions = [legit_session(customers[i], now, hard=rng.random() < 0.01) for i in picks]
    if rng.random() < 0.3:  # keep fraud visible in the live demo
        sessions.append(ato_session(fraudsters[int(rng.integers(len(fraudsters)))], now))
    for s in sessions:
        # shift each session so it ends "now" (no future timestamps)
        shift = s[-1][1] - now + timedelta(seconds=float(rng.uniform(0, LIVE_INTERVAL_S)))
        evs += [(st, ts - shift, {**p, **({"ts": iso(ts - shift)} if st == "transfers" else {"event_ts": iso(ts - shift)})}) for st, ts, p in s]
    return evs

if MODE in ("live", "backfill_and_live"):
    # Same customers (seeded by SEED), new event stream: re-seed the shared rng used by uid()/sessions.
    rng = np.random.default_rng(LIVE_SEED)
    print("live_seed:", LIVE_SEED)
    end, ticks, totals = time.time() + LIVE_DURATION_S, 0, defaultdict(int)
    while time.time() < end:
        t_start = time.time()
        stats = write_batch(live_tick(datetime.now(timezone.utc)), f"live_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}")
        for k, v in stats.items():
            totals[k] += v
        ticks += 1
        time.sleep(max(0, LIVE_INTERVAL_S - (time.time() - t_start)))
    print(f"Live mode: {ticks} micro-batches, records={dict(totals)}")

In [0]:
summary = []
for s in STREAMS:
    df = spark.read.json(f"{VOL_ROOT}/{s}").select("*", F.col("_metadata.file_path").alias("_file"))
    summary.append(df.agg(F.lit(s).alias("stream"), F.count("*").alias("records"),
                          F.countDistinct("sequenceNumber").alias("distinct_seq"),
                          F.countDistinct("data.customer_id").alias("customers"),
                          F.sum(F.col("data.customer_id").isNull().cast("int")).alias("null_customer"),
                          F.countDistinct("_file").alias("files")))
from functools import reduce
display(reduce(lambda a, b: a.unionByName(b), summary))

tr = spark.read.json(f"{VOL_ROOT}/transfers").select("data.*").dropDuplicates(["transfer_id"])
display(tr.groupBy("is_fraud").agg(F.count("*").alias("transfers"), F.countDistinct("customer_id").alias("customers"),
                                    F.round(F.avg("amount_usd"), 2).alias("avg_amount"),
                                    F.round(F.avg((F.col("payout_method").isin("cash_pickup", "mobile_wallet")).cast("int")), 3).alias("pct_cash_or_wallet"))
          .withColumn("share", F.round(F.col("transfers") / F.lit(tr.count()), 4)))
display(spark.read.json(f"{VOL_ROOT}/otp").limit(3))